In [2]:
import os
import yaml
import copy

import numpy as np
import pandas as pd
import xarray as xr
import xesmf as xe

In [4]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
varnames = [
    'total_column_water',
    'surface_pressure',
    '2m_temperature',
    'minimum_2m_temperature_since_previous_post_processing', 
    'maximum_2m_temperature_since_previous_post_processing', 
    '2m_dewpoint_temperature',
    '10m_u_component_of_wind',
    '10m_v_component_of_wind',
    'total_precipitation',
    'surface_solar_radiation_downwards',
    'surface_thermal_radiation_downwards'
]

## Domain subsetting around stations

In [9]:
ds_static = xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/static/static.zarr')
lon = ds_static['lon'].values
lat = ds_static['lat'].values

In [ ]:
dict_loc = {
    'Pituffik': (76.4, -68.575),
    'Fairbanks': (64.75, -147.4),
    'Guam': (13.475, 144.75),
    'Yuma_PG': (33.125, -114.125),
    'Fort_Bragg': (35.05, -79.115),
}

stn_pad = 10
keys = list(dict_loc.keys())

base_dir = '/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/'
years = np.arange(1958, 2026)

def _wrap_lon_180(lon):
    """Map longitude to [-180, 180). Works for DataArray/ndarray/scalar."""
    return ((lon + 180) % 360) - 180

def _infer_lat_lon_names(ds):
    # Adjust candidates if your ERA5 zarr uses different naming
    lat_name = 'lat' if 'lat' in ds.coords else ('latitude' if 'latitude' in ds.coords else None)
    lon_name = 'lon' if 'lon' in ds.coords else ('longitude' if 'longitude' in ds.coords else None)
    if lat_name is None or lon_name is None:
        raise KeyError(f"Cannot find lat/lon coords. Found coords: {list(ds.coords)}")
    return lat_name, lon_name

In [11]:
for i, key in enumerate(keys):
    stn_lat, stn_lon = dict_loc[key]
    lon_min, lon_max = stn_lon - stn_pad, stn_lon + stn_pad
    lat_min, lat_max = stn_lat - stn_pad, stn_lat + stn_pad

    out_dir = os.path.join(base_dir, key)
    os.makedirs(out_dir, exist_ok=True)

    for year in years:
        fn = os.path.join(base_dir, f'ERA5_{year}.zarr')
        ds = xr.open_zarr(fn)

        lat_name, lon_name = _infer_lat_lon_names(ds)

        # ---- longitude handling (supports 0..360 or -180..180, and dateline crossing) ----
        lon = ds[lon_name]
        lon_is_360 = (lon.min() >= 0) and (lon.max() > 180)

        if lon_is_360:
            # Keep in [0, 360)
            lon_min_mod = lon_min % 360
            lon_max_mod = lon_max % 360

            if lon_min_mod <= lon_max_mod:
                lon_mask = (lon >= lon_min_mod) & (lon <= lon_max_mod)
            else:
                # crosses 0 meridian in 0..360 system
                lon_mask = (lon >= lon_min_mod) | (lon <= lon_max_mod)
        else:
            # Use [-180, 180)
            lon_wrapped = _wrap_lon_180(lon)
            lon_min_w = _wrap_lon_180(lon_min)
            lon_max_w = _wrap_lon_180(lon_max)

            if lon_min_w <= lon_max_w:
                lon_mask = (lon_wrapped >= lon_min_w) & (lon_wrapped <= lon_max_w)
            else:
                # crosses dateline in -180..180 system
                lon_mask = (lon_wrapped >= lon_min_w) | (lon_wrapped <= lon_max_w)

        lat_mask = (ds[lat_name] >= lat_min) & (ds[lat_name] <= lat_max)

        # ---- subset ----
        ds_sub = ds.sel({lat_name: lat_mask, lon_name: lon_mask})

        # Ensure coords are monotonically increasing for nicer downstream behavior
        if ds_sub[lat_name][0] > ds_sub[lat_name][-1]:
            ds_sub = ds_sub.sortby(lat_name)

        # ---- save ----
        save_name = os.path.join(out_dir, f'ERA5_{key}_{year}.zarr')
        print(save_name)
        ds_sub.to_zarr(save_name, mode='w')

/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1958.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1959.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1960.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1961.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1962.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1963.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1964.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1965.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1966.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1967.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1968.zarr
/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Pituffik/ERA5_Pituffik_1969.zarr
/glade/derecho/scratch/ksha/

In [12]:
xr.open_zarr('/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/Fort_Bragg/ERA5_Fort_Bragg_2025.zarr')

<xarray.Dataset>
Dimensions:                                                (time: 365, lat: 21,
                                                            lon: 16)
Coordinates:
  * lat                                                    (lat) float64 25.9...
  * lon                                                    (lon) float64 271....
  * time                                                   (time) datetime64[ns] ...
Data variables:
    10m_u_component_of_wind                                (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
    10m_v_component_of_wind                                (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
    2m_dewpoint_temperature                                (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
    2m_temperature                                         (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
    maximum_2m_temperature_since_previous_post_processing  (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
    minimum_2m_temperature_since_previous_post_processing  (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
    surface_pressure                                       (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
    surface_solar_radiation_downwards                      (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
    surface_thermal_radiation_downwards                    (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
    total_column_water                                     (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
    total_precipitation                                    (time, lat, lon) float32 dask.array<chunksize=(32, 21, 16), meta=np.ndarray>
Attributes:
    regrid_method:  bilinear

In [13]:
lon_min, lon_max

(-89.115, -69.115)

In [14]:
lat_min, lat_max

(25.049999999999997, 45.05)